In [ ]:
from IPython.display import HTML, display
display(HTML('''
<div style="text-align:right;padding:8px 0;">
<button id="toggle-code-btn" style="
  padding:8px 18px;font-size:14px;cursor:pointer;
  background:#4a90d9;color:white;border:none;border-radius:5px;
  box-shadow:0 2px 4px rgba(0,0,0,0.2);
" onclick="
  var cells = document.querySelectorAll('.jp-Cell-inputWrapper');
  var btn = document.getElementById('toggle-code-btn');
  var show = btn.dataset.show !== 'true';
  btn.dataset.show = show;
  btn.textContent = show ? '\u9690\u85cf\u4ee3\u7801 \ud83d\udd12' : '\u663e\u793a\u4ee3\u7801 \ud83d\udd13';
  cells.forEach(function(c){ c.style.display = show ? '' : 'none'; });
">\u663e\u793a\u4ee3\u7801 \ud83d\udd13</button>
</div>
'''))


# 第2章 · 2.1 网络基础与拓扑 (Network Fundamentals & Topologies)
> Cambridge International AS & A Level Computer Science (9618)

---

本 notebook 将从零开始介绍：
1. 为什么需要网络？
2. LAN 与 WAN 的区别
3. Client-Server 与 Peer-to-Peer 模型
4. Thin Client 与 Thick Client
5. 网络拓扑：Bus、Star、Mesh、Hybrid
6. 练习题

## 1. 为什么需要网络？(Why do networks exist?)

### 1.1 想象一个没有网络的世界

假设你的学校有 30 台电脑，但没有任何网络连接：
- 想和同学共享一个文件？你只能用 **U盘** 拷贝，走到他的电脑旁边。
- 想打印一份作业？每台电脑都需要连一台自己的打印机——30 台打印机！
- 老师想把作业发给所有人？需要一台一台地拷贝。

这就是 **独立计算机 (Standalone Computer)** 的世界——低效、浪费资源、难以协作。

### 1.2 网络解决了什么问题？

| 问题 | 没有网络 | 有了网络 |
|------|----------|----------|
| 文件共享 | U盘传递，慢且不安全 | 直接通过网络传输，秒级完成 |
| 硬件共享 | 每台电脑配打印机 | 30台电脑共享1台网络打印机 |
| 通信 | 走到对方电脑旁 | Email、即时消息、视频通话 |
| 数据备份 | 每台电脑自己备份 | 集中备份到服务器 |
| 软件管理 | 每台电脑单独安装 | 集中部署和更新 |

### 1.3 网络的定义

> **计算机网络 (Computer Network)** = 两台或多台计算设备通过传输介质连接在一起，能够进行数据交换和资源共享的系统。

关键要素：
- **节点 (Node)**：网络中的任何设备（电脑、打印机、手机）
- **传输介质 (Transmission Medium)**：连接设备的"路"（网线、WiFi信号等）
- **协议 (Protocol)**：设备之间"说话的规则"（就像人需要用同一种语言交流）

## 2. LAN vs WAN：局域网与广域网

### 2.1 LAN (Local Area Network) — 局域网

**LAN** 覆盖一个 **小的地理区域**，比如：
- 你家里的 WiFi 网络
- 学校的机房
- 一家公司的办公室

**特点：**
- 覆盖范围小（通常一栋楼或一个校园内）
- 传输速度快（通常 100 Mbps ~ 10 Gbps）
- 由组织自己拥有和管理
- 使用私有的硬件和线路

### 2.2 WAN (Wide Area Network) — 广域网

**WAN** 覆盖一个 **大的地理区域**，比如：
- 一家公司在北京和上海的办公室之间的网络
- **互联网 (Internet)** 就是世界上最大的 WAN！

**特点：**
- 覆盖范围大（城市之间、国家之间、甚至全球）
- 传输速度相对较慢（受距离影响）
- 通常需要租用电信公司的线路
- 由多个组织共同维护

### 2.3 对比总结

| 特性 | LAN | WAN |
|------|-----|-----|
| 地理范围 | 小（一栋楼/校园） | 大（城市/国家/全球） |
| 传输速度 | 快（100Mbps-10Gbps） | 相对慢 |
| 谁拥有 | 组织自己 | 通常租用电信线路 |
| 成本 | 初始建设成本 | 持续的租赁费用 |
| 错误率 | 低 | 较高（距离远干扰多） |
| 例子 | 学校机房、家庭WiFi | Internet、银行全国网络 |

> **类比**：LAN 就像你家里的对讲机（范围小、速度快、自己买的），WAN 就像电话网络（范围大、要付话费、电信公司维护）。

In [ ]:
%matplotlib inline

# === 中文字体配置 (Chinese Font Setup) ===
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')

# 方法1: 全局设置 WenQuanYi Zen Hei 字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 方法2: 强制重建字体缓存（首次运行可能需要）
fm._load_fontmanager(try_read_cache=False)

# 验证
test_font = fm.findfont('WenQuanYi Zen Hei')
if 'WenQuanYi Zen Hei' in test_font or 'wqy' in test_font.lower():
    print('✅ 中文字体 WenQuanYi Zen Hei 已加载')
else:
    print(f'⚠️ WenQuanYi Zen Hei 未找到，使用: {test_font}')
    # Fallback: 直接注册字体文件
    font_path = '/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc'
    if __import__('os').path.exists(font_path):
        fm.fontManager.addfont(font_path)
        plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei'] + plt.rcParams['font.sans-serif']
        print('✅ 已手动加载 WenQuanYi Zen Hei 字体文件')

# 可视化：LAN vs WAN 的概念图
import matplotlib.patches as mpatches


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LAN 示意图
ax1 = axes[0]
ax1.set_xlim(0, 10)
ax1.set_ylim(0, 8)
ax1.set_title('LAN (局域网) - 学校机房', fontsize=14, fontweight='bold')
ax1.set_aspect('equal')
ax1.axis('off')

# 画一个建筑物框
building = mpatches.FancyBboxPatch((0.5, 0.5), 9, 7, boxstyle='round,pad=0.3',
                                     facecolor='#E3F2FD', edgecolor='#1565C0', linewidth=2)
ax1.add_patch(building)
ax1.text(5, 7.0, 'School Building', ha='center', fontsize=11, color='#1565C0', fontweight='bold')

# 电脑节点
positions = [(2, 5), (5, 5), (8, 5), (2, 2.5), (5, 2.5), (8, 2.5)]
for i, (x, y) in enumerate(positions):
    circle = plt.Circle((x, y), 0.5, color='#42A5F5', ec='#1565C0', lw=2)
    ax1.add_patch(circle)
    ax1.text(x, y, f'PC{i+1}', ha='center', va='center', fontsize=8, color='white', fontweight='bold')

# Switch 在中间
rect = mpatches.FancyBboxPatch((4.2, 3.5), 1.6, 0.7, boxstyle='round,pad=0.1',
                                 facecolor='#FF7043', edgecolor='#D84315', linewidth=2)
ax1.add_patch(rect)
ax1.text(5, 3.85, 'Switch', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# 连线
for x, y in positions:
    ax1.plot([x, 5], [y, 3.85], 'b-', lw=1, alpha=0.5)

# WAN 示意图
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 8)
ax2.set_title('WAN (广域网) - 跨城市连接', fontsize=14, fontweight='bold')
ax2.set_aspect('equal')
ax2.axis('off')

# 北京
bj = mpatches.FancyBboxPatch((0.5, 4.5), 3, 2.5, boxstyle='round,pad=0.2',
                               facecolor='#E8F5E9', edgecolor='#2E7D32', linewidth=2)
ax2.add_patch(bj)
ax2.text(2, 6.5, 'Beijing LAN', ha='center', fontsize=10, color='#2E7D32', fontweight='bold')
for x, y in [(1.3, 5.3), (2, 5.3), (2.7, 5.3)]:
    c = plt.Circle((x, y), 0.3, color='#66BB6A', ec='#2E7D32', lw=1.5)
    ax2.add_patch(c)

# 上海
sh = mpatches.FancyBboxPatch((6.5, 4.5), 3, 2.5, boxstyle='round,pad=0.2',
                               facecolor='#FFF3E0', edgecolor='#E65100', linewidth=2)
ax2.add_patch(sh)
ax2.text(8, 6.5, 'Shanghai LAN', ha='center', fontsize=10, color='#E65100', fontweight='bold')
for x, y in [(7.3, 5.3), (8, 5.3), (8.7, 5.3)]:
    c = plt.Circle((x, y), 0.3, color='#FFA726', ec='#E65100', lw=1.5)
    ax2.add_patch(c)

# Router
for cx, cy, label in [(3.8, 3.0, 'Router'), (6.2, 3.0, 'Router')]:
    r = mpatches.FancyBboxPatch((cx-0.5, cy-0.3), 1, 0.6, boxstyle='round,pad=0.1',
                                  facecolor='#CE93D8', edgecolor='#6A1B9A', linewidth=2)
    ax2.add_patch(r)
    ax2.text(cx, cy, label, ha='center', va='center', fontsize=8, color='white', fontweight='bold')

# WAN link
ax2.annotate('', xy=(5.7, 3.0), xytext=(4.3, 3.0),
             arrowprops=dict(arrowstyle='<->', color='red', lw=2.5))
ax2.text(5, 3.4, 'WAN Link\n(Leased Line)', ha='center', fontsize=9, color='red')

# connect LANs to routers
ax2.plot([2, 3.8], [5.3, 3.3], 'g--', lw=1.5)
ax2.plot([8, 6.2], [5.3, 3.3], color='orange', ls='--', lw=1.5)

# Cloud
cloud = mpatches.FancyBboxPatch((3.5, 0.5), 3, 1.5, boxstyle='round,pad=0.3',
                                  facecolor='#E0E0E0', edgecolor='#616161', linewidth=2)
ax2.add_patch(cloud)
ax2.text(5, 1.25, 'Internet (WAN)', ha='center', va='center', fontsize=10, color='#424242', fontweight='bold')

ax2.plot([3.8, 4.5], [2.7, 2.0], color='gray', ls=':', lw=1.5)
ax2.plot([6.2, 5.5], [2.7, 2.0], color='gray', ls=':', lw=1.5)

plt.tight_layout()
plt.savefig('lan_vs_wan.png', dpi=100, bbox_inches='tight')
plt.show()
print('LAN 在一个建筑物内，设备直接连接到 Switch。')
print('WAN 通过 Router 和租用线路连接不同城市的 LAN。')


## 3. Client-Server vs Peer-to-Peer 网络模型

### 3.1 Client-Server 模型（客户端-服务器模型）

**核心思想**：有一台（或多台）强大的 **服务器 (Server)** 集中管理资源，其他设备作为 **客户端 (Client)** 向服务器请求服务。

**工作方式：**
1. 客户端发送 **请求 (Request)** 给服务器
2. 服务器处理请求
3. 服务器返回 **响应 (Response)** 给客户端

**真实例子：**
- 你打开百度搜索 → 你的浏览器是 Client，百度的服务器是 Server
- 你用学校的电脑登录 → 你的电脑是 Client，学校的认证服务器是 Server
- 你玩在线游戏 → 你的电脑/手机是 Client，游戏公司的服务器是 Server

**优点：**
- 集中管理：备份、安全、用户权限都在服务器统一控制
- 资源共享：文件、打印机等集中管理
- 安全性高：服务器可以设置统一的安全策略

**缺点：**
- 服务器故障 = 整个网络瘫痪（**单点故障 Single Point of Failure**）
- 服务器成本高（需要强大的硬件）
- 需要专业人员维护

### 3.2 Peer-to-Peer (P2P) 模型（对等网络模型）

**核心思想**：没有专门的服务器！每台电脑 **既是客户端又是服务器**，大家地位平等。

**真实例子：**
- BitTorrent 下载：每个用户既下载文件也上传文件给别人
- AirDrop/蓝牙传文件：两台设备直接通信
- 区块链/比特币：没有中央服务器，所有节点平等

**优点：**
- 没有单点故障（一台坏了不影响其他人）
- 成本低（不需要购买昂贵的服务器）
- 设置简单

**缺点：**
- 难以集中管理
- 安全性较差（每台电脑自己负责安全）
- 数据不一致（没有统一的数据源）
- 随着节点增多，性能下降

In [ ]:
# 可视化：Client-Server vs Peer-to-Peer
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# ── Client-Server ──
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title('Client-Server Model', fontsize=14, fontweight='bold')
ax.set_aspect('equal')
ax.axis('off')

# Server
server = mpatches.FancyBboxPatch((3.5, 7.5), 3, 1.5, boxstyle='round,pad=0.2',
                                   facecolor='#EF5350', edgecolor='#B71C1C', linewidth=2)
ax.add_patch(server)
ax.text(5, 8.25, 'Server', ha='center', va='center', fontsize=12, color='white', fontweight='bold')

# Clients
client_positions = [(1, 2), (3.5, 2), (6, 2), (8.5, 2)]
for i, (x, y) in enumerate(client_positions):
    rect = mpatches.FancyBboxPatch((x-0.6, y-0.4), 1.2, 0.8, boxstyle='round,pad=0.1',
                                     facecolor='#42A5F5', edgecolor='#1565C0', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, f'Client {i+1}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    # Arrows: request up, response down
    ax.annotate('', xy=(x-0.1, 7.5), xytext=(x-0.1, 2.4),
                arrowprops=dict(arrowstyle='->', color='#1565C0', lw=1.5))
    ax.annotate('', xy=(x+0.1, 2.4), xytext=(x+0.1, 7.5),
                arrowprops=dict(arrowstyle='->', color='#EF5350', lw=1.5))

ax.text(0.5, 5, 'Request', fontsize=10, color='#1565C0', rotation=90, va='center')
ax.text(9.5, 5, 'Response', fontsize=10, color='#EF5350', rotation=90, va='center')

# ── Peer-to-Peer ──
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 10)
ax2.set_title('Peer-to-Peer Model', fontsize=14, fontweight='bold')
ax2.set_aspect('equal')
ax2.axis('off')

# Peers in circle
n_peers = 5
angles = np.linspace(0, 2*np.pi, n_peers, endpoint=False) - np.pi/2
cx, cy, r = 5, 5, 3
peer_pos = [(cx + r*np.cos(a), cy + r*np.sin(a)) for a in angles]

for i, (x, y) in enumerate(peer_pos):
    rect = mpatches.FancyBboxPatch((x-0.7, y-0.4), 1.4, 0.8, boxstyle='round,pad=0.1',
                                     facecolor='#66BB6A', edgecolor='#2E7D32', linewidth=2)
    ax2.add_patch(rect)
    ax2.text(x, y, f'Peer {i+1}', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# Connect every peer to every other peer
for i in range(n_peers):
    for j in range(i+1, n_peers):
        x1, y1 = peer_pos[i]
        x2, y2 = peer_pos[j]
        ax2.plot([x1, x2], [y1, y2], 'g-', lw=1, alpha=0.4)
        mid_x, mid_y = (x1+x2)/2, (y1+y2)/2
        ax2.annotate('', xy=(mid_x + 0.15, mid_y + 0.15), xytext=(mid_x - 0.15, mid_y - 0.15),
                     arrowprops=dict(arrowstyle='<->', color='#2E7D32', lw=1))

ax2.text(5, 0.5, 'Every peer can communicate\nwith every other peer', ha='center',
         fontsize=10, style='italic', color='#2E7D32')

plt.tight_layout()
plt.savefig('cs_vs_p2p.png', dpi=100, bbox_inches='tight')
plt.show()

## 4. Thin Client vs Thick Client（瘦客户端 vs 胖客户端）

### 4.1 什么是 Thin Client？

**Thin Client（瘦客户端）** 是一种几乎不做任何本地处理的设备——所有的计算工作都在 **服务器** 上完成。

> **类比**：Thin Client 就像一个 **电视遥控器**——它本身不能播放任何内容，只是把你的操作传给电视（服务器），然后显示结果。

**特征：**
- 硬件配置低（不需要强大的 CPU 或大硬盘）
- 所有程序在服务器上运行
- 客户端只负责显示画面和接收输入
- **例子**：Google Chromebook（很多工作在云端完成）、远程桌面、网页版 Office

### 4.2 什么是 Thick Client？

**Thick Client（胖客户端）** 是一种在本地完成大部分处理工作的设备。

> **类比**：Thick Client 就像一个 **全功能的游戏主机**——它自己就能运行游戏，不依赖外部服务器。

**特征：**
- 需要强大的硬件（好的 CPU、大内存、大硬盘）
- 程序安装在本地并在本地运行
- 可以离线工作
- **例子**：你的笔记本电脑（装了 Word、Photoshop）、游戏 PC

### 4.3 对比

| 特性 | Thin Client | Thick Client |
|------|------------|-------------|
| 处理位置 | 服务器 | 本地 |
| 硬件要求 | 低 | 高 |
| 成本 | 低（设备便宜） | 高（设备昂贵） |
| 维护 | 容易（只更新服务器） | 难（每台都要更新） |
| 离线工作 | 不能 | 可以 |
| 网络依赖 | 强（没网就不能用） | 弱 |
| 安全性 | 高（数据在服务器） | 低（数据在本地） |
| 性能 | 取决于服务器和网络 | 取决于本地硬件 |

In [ ]:
# 可视化：Thin vs Thick Client
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Thin Client ──
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 8)
ax.set_title('Thin Client (瘦客户端)', fontsize=14, fontweight='bold')
ax.axis('off')

# Server (big)
s = mpatches.FancyBboxPatch((3, 5), 4, 2.3, boxstyle='round,pad=0.3',
                              facecolor='#EF5350', edgecolor='#B71C1C', linewidth=2.5)
ax.add_patch(s)
ax.text(5, 6.8, 'Server', ha='center', fontsize=12, color='white', fontweight='bold')
ax.text(5, 6.1, 'CPU | RAM | Storage', ha='center', fontsize=9, color='#FFCDD2')
ax.text(5, 5.5, 'All processing here', ha='center', fontsize=9, color='#FFCDD2')

# Thin clients (small)
for i, x in enumerate([1.5, 5, 8.5]):
    r = mpatches.FancyBboxPatch((x-0.8, 0.5), 1.6, 1.2, boxstyle='round,pad=0.1',
                                  facecolor='#90CAF9', edgecolor='#1565C0', linewidth=1.5)
    ax.add_patch(r)
    ax.text(x, 1.1, f'Thin {i+1}', ha='center', va='center', fontsize=9, color='#0D47A1', fontweight='bold')
    ax.text(x, 0.7, 'Display only', ha='center', fontsize=7, color='#1565C0')
    ax.annotate('', xy=(x, 1.7), xytext=(x, 5.0),
                arrowprops=dict(arrowstyle='<->', color='#1565C0', lw=1.5, ls='--'))

ax.text(5, 3.5, 'Screen output +\nkeyboard/mouse input', ha='center',
        fontsize=9, style='italic', color='#1565C0')

# ── Thick Client ──
ax2 = axes[1]
ax2.set_xlim(0, 10)
ax2.set_ylim(0, 8)
ax2.set_title('Thick Client (胖客户端)', fontsize=14, fontweight='bold')
ax2.axis('off')

# Server (small role)
s2 = mpatches.FancyBboxPatch((3.5, 5.5), 3, 1.5, boxstyle='round,pad=0.2',
                               facecolor='#FFCDD2', edgecolor='#E57373', linewidth=1.5)
ax2.add_patch(s2)
ax2.text(5, 6.6, 'Server', ha='center', fontsize=10, color='#C62828')
ax2.text(5, 5.9, 'File storage only', ha='center', fontsize=8, color='#E57373')

# Thick clients (big)
for i, x in enumerate([1.5, 5, 8.5]):
    r = mpatches.FancyBboxPatch((x-1.1, 0.5), 2.2, 2.5, boxstyle='round,pad=0.2',
                                  facecolor='#42A5F5', edgecolor='#1565C0', linewidth=2.5)
    ax2.add_patch(r)
    ax2.text(x, 2.3, f'Thick {i+1}', ha='center', fontsize=10, color='white', fontweight='bold')
    ax2.text(x, 1.6, 'CPU | RAM', ha='center', fontsize=8, color='#BBDEFB')
    ax2.text(x, 1.1, 'Local processing', ha='center', fontsize=8, color='#BBDEFB')
    ax2.annotate('', xy=(x, 3.0), xytext=(x, 5.5),
                arrowprops=dict(arrowstyle='<->', color='gray', lw=1, ls=':'))

ax2.text(5, 4.3, 'Only file sync / updates', ha='center',
         fontsize=9, style='italic', color='gray')

plt.tight_layout()
plt.savefig('thin_vs_thick.png', dpi=100, bbox_inches='tight')
plt.show()

## 5. 网络拓扑 (Network Topologies)

### 什么是拓扑？

**网络拓扑 (Network Topology)** 描述的是网络中设备的 **排列方式** 和 **连接方式**。

> **类比**：拓扑就像城市的道路布局——环形、星形、网格形都会影响交通效率。

我们要学习四种主要拓扑：
1. **Bus Topology（总线拓扑）**
2. **Star Topology（星形拓扑）**
3. **Mesh Topology（网状拓扑）**
4. **Hybrid Topology（混合拓扑）**

### 5.1 Bus Topology（总线拓扑）

**结构**：所有设备连接到一条 **主干线缆 (Backbone Cable)**，线缆两端有 **终结器 (Terminator)** 防止信号反射。

**数据如何传输：**
1. 一台电脑发送数据，数据沿主干线缆向两个方向传播
2. **所有设备都能收到** 这个数据
3. 每台设备检查数据的目标地址——如果不是给自己的，就忽略

> **类比**：Bus 拓扑就像一条 **街道上的广播喇叭**——所有人都能听到，但只有名字被叫到的人才会回应。

**优点：**
- 结构简单，安装方便
- 使用的线缆少，成本低
- 适合小型临时网络

**缺点：**
- 主干线缆断了 → 整个网络瘫痪！
- 数据冲突（多台电脑同时发送数据会碰撞）
- 性能随设备增多而下降
- 安全性差（所有数据所有设备都能"看到"）
- 故障排查困难

In [ ]:
# 可视化：Bus Topology
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(1, 1, figsize=(12, 4))
ax.set_xlim(0, 14)
ax.set_ylim(0, 5)
ax.set_title('Bus Topology (总线拓扑)', fontsize=14, fontweight='bold')
ax.axis('off')

# Backbone
ax.plot([1, 13], [2.5, 2.5], 'k-', lw=4, solid_capstyle='round')
ax.text(0.3, 2.5, 'T', fontsize=14, fontweight='bold', color='red',
        ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='#FFCDD2', edgecolor='red'))
ax.text(13.7, 2.5, 'T', fontsize=14, fontweight='bold', color='red',
        ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='#FFCDD2', edgecolor='red'))

# Devices
positions_x = [2, 4.5, 7, 9.5, 12]
labels = ['PC1', 'PC2', 'PC3', 'Printer', 'PC4']
colors = ['#42A5F5', '#42A5F5', '#42A5F5', '#FF7043', '#42A5F5']

for x, label, c in zip(positions_x, labels, colors):
    ax.plot([x, x], [2.5, 3.5], 'k-', lw=2)
    rect = mpatches.FancyBboxPatch((x-0.6, 3.5), 1.2, 0.8, boxstyle='round,pad=0.1',
                                     facecolor=c, edgecolor='black', linewidth=1.5)
    ax.add_patch(rect)
    ax.text(x, 3.9, label, ha='center', va='center', fontsize=9, color='white', fontweight='bold')

# Data flow arrows
ax.annotate('', xy=(9, 2.0), xytext=(4.5, 2.0),
            arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=2.5))
ax.text(6.75, 1.5, 'Data flows in both directions', ha='center', fontsize=10,
        style='italic', color='#4CAF50')

ax.text(7, 0.5, 'T = Terminator (终结器) - prevents signal reflection',
        ha='center', fontsize=10, color='red')

plt.tight_layout()
plt.savefig('bus_topology.png', dpi=100, bbox_inches='tight')
plt.show()

### 5.2 Star Topology（星形拓扑）

**结构**：所有设备连接到一个 **中央设备 (Central Device)**——通常是 **Switch** 或 **Hub**。

**数据如何传输：**
1. 发送方把数据发给中央 Switch
2. Switch 根据目标地址，只把数据转发给目标设备
3. 其他设备不会收到这个数据（比 Bus 安全！）

> **类比**：Star 拓扑就像 **机场的航线图**——所有航线都经过中央枢纽（Hub），乘客通过枢纽转机到目的地。

**优点：**
- 一台设备故障不影响其他设备
- 容易添加或移除设备
- 性能好（数据只发给目标设备，不会广播给所有人）
- 故障排查容易（只需检查中央设备到故障设备的线路）

**缺点：**
- **中央设备故障 = 整个网络瘫痪**（又是单点故障！）
- 比 Bus 需要更多线缆（每台设备都要一条线连到中央）
- 中央设备的成本

> **Star 是目前最常用的 LAN 拓扑！** 你学校的机房几乎一定是 Star 拓扑。

In [ ]:
# 可视化：Star Topology
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_title('Star Topology (星形拓扑)', fontsize=14, fontweight='bold')
ax.set_aspect('equal')
ax.axis('off')

# Central switch
switch = plt.Circle((5, 5), 0.7, color='#FF7043', ec='#D84315', lw=3)
ax.add_patch(switch)
ax.text(5, 5, 'Switch', ha='center', va='center', fontsize=11, color='white', fontweight='bold')

# Devices around the switch
n_devices = 6
angles = np.linspace(0, 2*np.pi, n_devices, endpoint=False) - np.pi/2
radius = 3.5
labels = ['PC1', 'PC2', 'Server', 'PC3', 'Printer', 'PC4']
colors = ['#42A5F5', '#42A5F5', '#EF5350', '#42A5F5', '#66BB6A', '#42A5F5']

for i, (angle, label, c) in enumerate(zip(angles, labels, colors)):
    x = 5 + radius * np.cos(angle)
    y = 5 + radius * np.sin(angle)

    rect = mpatches.FancyBboxPatch((x-0.7, y-0.4), 1.4, 0.8,
                                     boxstyle='round,pad=0.1',
                                     facecolor=c, edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, label, ha='center', va='center', fontsize=9, color='white', fontweight='bold')

    # Line to center
    dx = 5 - x
    dy = 5 - y
    dist = np.sqrt(dx**2 + dy**2)
    # shorten line so it doesn't overlap shapes
    sx = x + 0.8 * dx/dist
    sy = y + 0.8 * dy/dist
    ex = 5 - 0.75 * dx/dist
    ey = 5 - 0.75 * dy/dist
    ax.plot([sx, ex], [sy, ey], 'k-', lw=2, alpha=0.6)

# Highlight data path
ax.annotate('', xy=(5, 5.7), xytext=(5-0.1, 5+radius-0.5),
            arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=2.5))
ax.annotate('', xy=(5+radius*np.cos(angles[3])+0.1, 5+radius*np.sin(angles[3])+0.5),
            xytext=(5, 4.3),
            arrowprops=dict(arrowstyle='->', color='#4CAF50', lw=2.5))

ax.text(5, 0.5, 'All data passes through the central Switch',
        ha='center', fontsize=11, style='italic', color='#D84315')

plt.tight_layout()
plt.savefig('star_topology.png', dpi=100, bbox_inches='tight')
plt.show()

### 5.3 Mesh Topology（网状拓扑）

**结构**：每台设备都与 **其他所有设备** 直接连接（Full Mesh），或者与 **部分设备** 直接连接（Partial Mesh）。

**数据如何传输：**
1. 数据可以通过 **多条路径** 到达目的地
2. 如果一条路径断了，可以自动切换到其他路径
3. 路由算法选择最佳路径

> **类比**：Full Mesh 就像一个 **朋友圈里每个人都互相加了好友**——任何两个人都可以直接聊天，不需要通过第三人转达。

**Full Mesh 的连接数量：**
- n 台设备需要 **n(n-1)/2** 条连接
- 5 台设备 = 5×4/2 = **10** 条连接
- 10 台设备 = 10×9/2 = **45** 条连接
- 100 台设备 = 100×99/2 = **4950** 条连接！

**优点：**
- **高可靠性**：多条冗余路径，一条断了还有其他的
- 没有单点故障
- 数据可以同时通过多条路径传输（高带宽）

**缺点：**
- **成本极高**：需要大量线缆和网络接口
- 配置和维护复杂
- 不适合大规模网络

> **实际使用**：Internet 的骨干网络就是 **Partial Mesh**（部分网状拓扑）。

In [ ]:
# 可视化：Full Mesh vs Partial Mesh
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (ax, title, partial) in enumerate(zip(axes,
    ['Full Mesh (完全网状)', 'Partial Mesh (部分网状)'],
    [False, True])):

    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_aspect('equal')
    ax.axis('off')

    n = 5
    angles = np.linspace(0, 2*np.pi, n, endpoint=False) - np.pi/2
    cx, cy, r = 5, 5, 3.2
    positions = [(cx + r*np.cos(a), cy + r*np.sin(a)) for a in angles]

    # Draw connections
    connection_count = 0
    for i in range(n):
        for j in range(i+1, n):
            if partial and ((i, j) in [(0, 3), (1, 4), (2, 4)]):
                continue  # Skip some connections for partial mesh
            x1, y1 = positions[i]
            x2, y2 = positions[j]
            ax.plot([x1, x2], [y1, y2], '-', lw=1.5, alpha=0.4,
                    color='#4CAF50' if not partial else '#FF9800')
            connection_count += 1

    # Draw nodes
    for i, (x, y) in enumerate(positions):
        circle = plt.Circle((x, y), 0.5, color='#42A5F5', ec='#1565C0', lw=2)
        ax.add_patch(circle)
        ax.text(x, y, f'N{i+1}', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

    ax.text(5, 0.5, f'Connections: {connection_count}', ha='center', fontsize=12,
            fontweight='bold', color='#333')

plt.tight_layout()
plt.savefig('mesh_topology.png', dpi=100, bbox_inches='tight')
plt.show()

# 计算 Full Mesh 连接数
print('Full Mesh 连接数公式: n(n-1)/2')
for n in [5, 10, 20, 50, 100]:
    connections = n * (n - 1) // 2
    print(f'  {n:3d} 台设备 -> {connections:5d} 条连接')

### 5.4 Hybrid Topology（混合拓扑）

**结构**：混合两种或多种基本拓扑。例如：
- Star-Bus：多个 Star 网络通过 Bus 主干连接
- Star-Mesh：核心用 Mesh 保证可靠性，边缘用 Star 降低成本

**真实例子：**
- 大型校园网络：每个教室是 Star 拓扑（连到一个 Switch），各教室的 Switch 再通过骨干线路连接
- 企业网络：总部和分部之间用 Mesh，每个办公室内部用 Star

**优点：**
- 灵活，可以根据需求选择最合适的拓扑组合
- 易于扩展

**缺点：**
- 设计和维护复杂
- 成本可能较高

In [ ]:
# 可视化：Hybrid Topology (Star-Bus)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(1, 1, figsize=(14, 7))
ax.set_xlim(0, 20)
ax.set_ylim(0, 10)
ax.set_title('Hybrid Topology: Star-Bus (混合拓扑示例)', fontsize=14, fontweight='bold')
ax.axis('off')

# Bus backbone
ax.plot([2, 18], [5, 5], 'k-', lw=4)
ax.text(1.3, 5, 'T', fontsize=12, fontweight='bold', color='red', ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='#FFCDD2', edgecolor='red'))
ax.text(18.7, 5, 'T', fontsize=12, fontweight='bold', color='red', ha='center', va='center',
        bbox=dict(boxstyle='round,pad=0.2', facecolor='#FFCDD2', edgecolor='red'))

# Three star clusters connected to backbone
star_centers = [(5, 5), (10, 5), (15, 5)]
star_labels = ['Classroom A', 'Classroom B', 'Classroom C']

for scx, scy in star_centers:
    # Switch on backbone
    switch = plt.Circle((scx, scy), 0.45, color='#FF7043', ec='#D84315', lw=2, zorder=5)
    ax.add_patch(switch)
    ax.text(scx, scy, 'SW', ha='center', va='center', fontsize=8, color='white', fontweight='bold', zorder=6)

    # Devices around each switch
    for angle in [np.pi/4, 3*np.pi/4, 5*np.pi/4, 7*np.pi/4]:
        dx = 1.8 * np.cos(angle)
        dy = 1.8 * np.sin(angle)
        px, py = scx + dx, scy + dy
        if 0.5 < px < 19.5 and 0.5 < py < 9.5:
            r = mpatches.FancyBboxPatch((px-0.4, py-0.3), 0.8, 0.6,
                                         boxstyle='round,pad=0.1',
                                         facecolor='#42A5F5', edgecolor='#1565C0', lw=1.5)
            ax.add_patch(r)
            ax.text(px, py, 'PC', ha='center', va='center', fontsize=7, color='white', fontweight='bold')
            ax.plot([scx, px], [scy, py], 'b-', lw=1, alpha=0.5)

for i, (scx, _) in enumerate(star_centers):
    ax.text(scx, 8.5, star_labels[i], ha='center', fontsize=11, fontweight='bold', color='#333')

ax.text(10, 0.5, 'Each classroom is a Star (connected to a Switch).\n'
        'All Switches are connected via a Bus backbone.',
        ha='center', fontsize=11, style='italic', color='#555')

plt.tight_layout()
plt.savefig('hybrid_topology.png', dpi=100, bbox_inches='tight')
plt.show()

### 5.5 拓扑对比总结

| 特性 | Bus | Star | Mesh | Hybrid |
|------|-----|------|------|--------|
| 结构 | 所有设备连到一条线 | 所有设备连到中央Switch | 设备之间互相连接 | 混合多种拓扑 |
| 成本 | 低 | 中等 | 高 | 取决于组合 |
| 可靠性 | 低（主干断=全断） | 中（中央故障=全断） | 高（冗余路径） | 取决于组合 |
| 扩展性 | 差 | 好 | 差（连接数爆炸） | 好 |
| 性能 | 差（共享带宽） | 好 | 好 | 取决于组合 |
| 故障排查 | 难 | 容易 | 复杂 | 中等 |
| 适用场景 | 小型临时网络 | 大多数 LAN | 关键基础设施 | 大型企业/校园 |

### 如何选择拓扑？

- **小型办公室/教室** → Star（最常用、最实用）
- **需要高可靠性的服务器集群** → Mesh
- **大型校园/企业** → Hybrid（Star + Bus/Mesh）
- **临时性小网络** → Bus（最简单、最便宜）

In [ ]:
# 互动：拓扑选择小测验
import random

scenarios = [
    ('一个有 20 台电脑的教室，需要共享一台打印机', 'Star', '教室适合 Star 拓扑：容易管理，一台电脑故障不影响其他电脑'),
    ('一个银行的数据中心，需要保证 24/7 不间断运行', 'Mesh', '银行需要高可靠性，Mesh 的冗余路径可以保证不中断'),
    ('临时搭建的会议网络，只需要用几个小时', 'Bus', '临时网络用 Bus 最简单、成本最低'),
    ('一个大学校园，有多栋教学楼', 'Hybrid', '大学用 Hybrid：每栋楼内部是 Star，楼与楼之间用骨干网络连接'),
    ('一个小型家庭网络，3台电脑+1台打印机', 'Star', '家庭网络通常通过 WiFi Router 形成 Star 拓扑'),
]

print('=' * 60)
print('  网络拓扑选择小测验')
print('=' * 60)

random.shuffle(scenarios)
score = 0
for i, (scenario, answer, explanation) in enumerate(scenarios[:3], 1):
    print(f'\n问题 {i}: {scenario}')
    print(f'  选项: A) Bus  B) Star  C) Mesh  D) Hybrid')
    print(f'  参考答案: {answer}')
    print(f'  解析: {explanation}')
    print('-' * 60)

## 6. 练习题 (Practice Exercises)

### 选择题

**Q1.** 以下哪个不是使用网络的优势？
- A) 共享打印机等硬件资源
- B) 更容易传播计算机病毒
- C) 方便文件共享
- D) 集中备份数据

**Q2.** LAN 和 WAN 的主要区别是什么？
- A) 使用的协议不同
- B) 覆盖的地理范围不同
- C) 连接的设备数量不同
- D) 使用的编程语言不同

**Q3.** 在 Star 拓扑中，如果中央 Switch 故障了，会发生什么？
- A) 只有连接到 Switch 的第一台设备受影响
- B) 整个网络都无法工作
- C) 网络会自动切换到 Bus 拓扑
- D) 不会有任何影响

**Q4.** 在 Client-Server 模型中，以下描述哪个是正确的？
- A) 所有设备地位平等
- B) 不需要专门的服务器
- C) 客户端向服务器发送请求，服务器返回响应
- D) 每台电脑既是客户端也是服务器

**Q5.** Full Mesh 拓扑中，10 台设备需要多少条连接？
- A) 10
- B) 20
- C) 45
- D) 90

---

### 参考答案
1. **B** — 病毒传播是网络的 *缺点*，不是优势
2. **B** — LAN 覆盖小区域，WAN 覆盖大区域
3. **B** — Star 拓扑的中央设备是单点故障
4. **C** — 这是 Client-Server 模型的核心工作方式
5. **C** — n(n-1)/2 = 10×9/2 = 45

### 简答题

**Q6.** 解释为什么 Star 拓扑是现在大多数 LAN 最常用的拓扑。（4分）

<details>
<summary>点击查看参考答案</summary>

- 易于管理和维护：添加/移除设备不影响其他设备（1分）
- 故障隔离好：一台设备故障不会影响网络其他部分（1分）
- 性能好：Switch 可以智能转发数据，避免不必要的广播（1分）
- 扩展性好：只需在 Switch 上添加新端口即可（1分）

</details>

**Q7.** 比较 Thin Client 和 Thick Client，分别给出一个适合使用的场景。（4分）

<details>
<summary>点击查看参考答案</summary>

**Thin Client**：
- 所有处理在服务器端完成，客户端只负责显示（1分）
- 适合场景：图书馆的公用电脑——只需要浏览网页和查询，不需要高性能本地处理（1分）

**Thick Client**：
- 大部分处理在本地完成（1分）
- 适合场景：视频编辑工作站——需要强大的本地 CPU 和 GPU 来处理视频渲染（1分）

</details>

---
*本节结束！下一节我们将学习网络硬件与传输介质。*